# Evaluate spectral predictions and concentration readouts

Compare stage-2 and stage-3 predictions with measured spectra. Report metal readouts and background behavior, fit calibration curves on clean rows within the test table, and export predictions and figures.

Readout gain is a standard-deviation ratio; a ratio near one alone does not establish correct calibration. Repeatability estimates provide measurement context, not a proof of optimal model performance. The shared evaluation functions are absent with `spice_moe.py`.

The separately included archived result tables can be inspected without the missing module. They are historical outputs, not results reproduced during this repository preparation.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ["MOE_PROJECT_ROOT"]) if "MOE_PROJECT_ROOT" in os.environ else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
from project_paths import configure_run
DATA_ROOT, RUN_ROOT, PREPARED_DIR, CHECKPOINT_DIR = configure_run(new=False)


In [ ]:
# === 05 / Step 0: setup, load testing set ================================
import os, sys
import numpy as np, pandas as pd, h5py, tensorflow as tf
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
import spice_moe as S

DATA_DIR = str(DATA_ROOT)
H5_DIR   = str(PREPARED_DIR)
CKPT_DIR = str(CHECKPOINT_DIR)
PLOT_DIR = str(RUN_ROOT / "figures")
SPEC_OUT = str(RUN_ROOT / "spectral_outputs")
for d in (PLOT_DIR, SPEC_OUT): os.makedirs(d, exist_ok=True)

def load_testing():
    with h5py.File(os.path.join(H5_DIR, "SPICE_testing_data.h5"), "r") as f:
        X = f["X_with_meta"][:]; cols = [c.decode() for c in f["X_columns"][:]]
    with h5py.File(os.path.join(H5_DIR, "SPICE_testing_label.h5"), "r") as f:
        Y = f["Y_with_meta"][:]
    cond  = X[:, :S.N_META].astype(np.float32)
    realp = (X[:, S.N_META:] / S.SPEC_SCALE).astype(np.float32)
    inf   = (Y[:, S.N_META:] / S.SPEC_SCALE).astype(np.float32)
    wl    = np.array([float(c) for c in cols[S.N_META:]])
    return cond, realp, inf, wl

cond_t, realp_t, inf_t, wl = load_testing()
SPEC_DIM = realp_t.shape[1]
ridx  = S.build_readout_index(wl)
bmask = S.background_mask(wl)
print("testing rows:", len(cond_t), " SPEC_DIM:", SPEC_DIM)

# The measurement's own repeatability, measured from the replicate structure of the
# testing labels.  This is the number every model error should be compared against:
# repeatability provides context for the model error, not a hard optimality certificate.
gk_t = S.condition_key(cond_t)
floors = S.measure_readout_floor(inf_t, ridx, gk_t)
print("measured read-out repeatability (ppm):", {k: round(v, 3) for k, v in floors.items()})
print("configured READOUT_FLOOR_PPM        :", S.READOUT_FLOOR_PPM)


In [ ]:
# === 05 / Step 1: load the trained stages and predict ====================
STAGES = {"stage2": ["Na", "Ca", "K"], "stage3": ["Na", "Ca", "K", "Mg"]}
pred_t = {}
for st, ions in STAGES.items():
    if not os.path.exists(os.path.join(CKPT_DIR, st)):
        print(f"skip {st} (no checkpoint)"); continue
    parts, model = S.build_stage(SPEC_DIM, ions, CKPT_DIR, st, name=f"moe_{st}")
    res = S.predict_residual(model, realp_t, cond_t, batch=64)
    pred_t[st] = np.clip(realp_t + res, 0.0, 1.0)
    print(f"{st}: predicted {pred_t[st].shape}")


In [ ]:
# === 05 / Step 2: the v6 report — metals AND background ==================
# v5 only ever reported the three metal read-outs.  The background is reported here
# because it is the CAUSE, not a side issue: the read-out is a peak height measured
# against the background and divided by the norm line, so a 1% background error
# becomes a systematic concentration error.
for st, pr in pred_t.items():
    S.full_report(cond_t, pr, inf_t, ridx, bmask, title=f"TESTING — {st}")


In [ ]:
# === 05 / Step 3: calibration curves -> concentration CSVs ===============
# Calibration is fitted on the CLEAN testing rows (all interferents = 0), then the
# same curve is applied to the label and to the prediction.  model_err is the only
# column that judges the model: it compares prediction with measurement, both read
# through the identical calibration.
clean = (cond_t[:, [S.idx_map[c] for c in S.BKG_COLS]] == 0).all(axis=1)
CAL = {}
for j, el in enumerate(('Cu', 'Ni', 'Zn')):
    r = S.np_peak_readout(inf_t[clean], ridx)[:, j]
    c = cond_t[clean, S.idx_map[el]]
    A = np.polyfit(c, r, 1); CAL[el] = (float(A[0]), float(A[1]))
    print(f"calibration {el}: readout = {A[0]:.4f} * ppm + {A[1]:.4f}")
pd.DataFrame([{"element": e, "slope": s, "intercept": i} for e, (s, i) in CAL.items()]
             ).to_csv(os.path.join(RUN_ROOT, "calibration_slope_v6.csv"), index=False)

def to_conc(spec):
    r = S.np_peak_readout(spec, ridx)
    return np.stack([(r[:, j] - CAL[el][1]) / CAL[el][0]
                     for j, el in enumerate(('Cu', 'Ni', 'Zn'))], axis=1)

for st, pr in pred_t.items():
    cl, cp = to_conc(inf_t), to_conc(pr)
    out = {c: cond_t[:, S.idx_map[c]] for c in ('Cu', 'Ni', 'Zn', 'Na', 'Ca', 'K', 'Mg')}
    for j, el in enumerate(('Cu', 'Ni', 'Zn')):
        out[f'{el}_true_ppm']   = cond_t[:, S.idx_map[el]]
        out[f'{el}_conc_label'] = cl[:, j]
        out[f'{el}_conc_pred']  = cp[:, j]
        out[f'{el}_model_err']  = cp[:, j] - cl[:, j]
    df = pd.DataFrame(out)
    df.to_csv(os.path.join(RUN_ROOT, f"calibration_result_v6_{st}.csv"), index=False)
    print(f"\n{st}: " + "  ".join(
        f"{el} MAE={df[f'{el}_model_err'].abs().mean():.3f} "
        f"bias={df[f'{el}_model_err'].mean():+.3f}" for el in ('Cu', 'Ni', 'Zn')))


In [ ]:
# === 05 / Step 4: export spectra =========================================
def to_spectrum_df(cond, spec_norm):
    d = pd.DataFrame(cond, columns=S.META_COLS)
    sp = pd.DataFrame(S.finalize_fake_spectrum(spec_norm) * S.SPEC_SCALE,
                      columns=[f"{w:.3f}" for w in wl])
    return pd.concat([d, sp], axis=1)

to_spectrum_df(cond_t, inf_t).to_csv(os.path.join(SPEC_OUT, "testing_label_spectrum.csv"), index=False)
for st, pr in pred_t.items():
    to_spectrum_df(cond_t, pr).to_csv(
        os.path.join(SPEC_OUT, f"testing_pred_spectrum_{st}.csv"), index=False)
print("spectra written to", SPEC_OUT)


In [ ]:
# === 05 / Step 5: validation-set evaluation ==============================
# Identical construction to training and testing: one clean replicate in, one
# measured replicate as the label, no averaging and no synthetic noise.  Because
# whole (Cu,Ni,Zn) combinations were held out, these combinations were never seen.
PLANS = ["na_train", "ca_train", "k_train", "mg_train",
         "naca_train", "nacak_train", "nacakmg_train"]
Pv = S.merge_plans([S.load_plan(H5_DIR, n) for n in PLANS])
_, va_idx = S.split_plan(Pv)
sub = va_idx[:8000]

for st, ions in STAGES.items():
    if st not in pred_t: continue
    parts, model = S.build_stage(SPEC_DIM, ions, CKPT_DIR, st, name=f"val_{st}")
    cond_v, pred_v, lab_v = S.predict_on_plan(model, Pv, sub)
    S.full_report(cond_v, pred_v, lab_v, ridx, bmask, title=f"VALIDATION — {st}")

    cl, cp = to_conc(lab_v), to_conc(pred_v)
    out = {c: cond_v[:, S.idx_map[c]] for c in ('Cu', 'Ni', 'Zn', 'Na', 'Ca', 'K', 'Mg')}
    for j, el in enumerate(('Cu', 'Ni', 'Zn')):
        out[f'{el}_conc_label'] = cl[:, j]
        out[f'{el}_conc_pred']  = cp[:, j]
        out[f'{el}_model_err']  = cp[:, j] - cl[:, j]
    pd.DataFrame(out).to_csv(
        os.path.join(RUN_ROOT, f"calibration_result_v6_val_{st}.csv"), index=False)
    to_spectrum_df(cond_v[:400], pred_v[:400]).to_csv(
        os.path.join(SPEC_OUT, f"val_pred_spectrum_{st}.csv"), index=False)
    to_spectrum_df(cond_v[:400], lab_v[:400]).to_csv(
        os.path.join(SPEC_OUT, f"val_label_spectrum_{st}.csv"), index=False)


In [ ]:
# === 05 / Step 6: spectra plots (full range + band zooms) ================
BANDS = [("Zn", S.ZN_RANGE), ("Ni", S.NI_RANGE), ("Cu", S.CU_RANGE)]
fi = lambda v: int(np.abs(wl - v).argmin())

def plot_case(st, i, save=None):
    pr, lb = pred_t[st][i], inf_t[i]
    c = {k: cond_t[i, S.idx_map[k]] for k in ('Cu','Ni','Zn','Na','Ca','K','Mg')}
    fig, ax = plt.subplots(2, 3, figsize=(12, 6),
                           gridspec_kw={"height_ratios": [1.5, 1]})
    g = fig.add_subplot(2, 1, 1)
    for a in ax[0]: a.remove()
    g.plot(wl, lb, "k", lw=1, label="measured")
    g.plot(wl, pr, "r--", lw=1, label="predicted")
    for nm, (lo, hi) in BANDS: g.axvspan(lo, hi, color="gray", alpha=0.15)
    g.set_xlim(200, 530); g.legend(); g.grid(alpha=0.3)
    g.set_title(", ".join(f"{k}={v:.0f}" for k, v in c.items()), fontsize=10)
    g.set_ylabel("normalized intensity")
    for j, (nm, (lo, hi)) in enumerate(BANDS):
        a, b = fi(lo), fi(hi)
        ax[1][j].plot(wl[a:b], lb[a:b], "k"); ax[1][j].plot(wl[a:b], pr[a:b], "r--")
        ax[1][j].set_title(nm, fontsize=9); ax[1][j].grid(alpha=0.3)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110, bbox_inches="tight"); plt.close(fig)
    else: plt.show()

st = "stage3" if "stage3" in pred_t else list(pred_t)[0]
worst = np.argsort(-np.abs(S.np_readout_ppm(pred_t[st], ridx)[:, 0]
                           - S.np_readout_ppm(inf_t, ridx)[:, 0]))[:3]
print("showing the 3 worst Cu cases for", st)
for i in worst: plot_case(st, int(i))

d = os.path.join(PLOT_DIR, f"testing_{st}"); os.makedirs(d, exist_ok=True)
for i in range(0, len(cond_t), max(1, len(cond_t)//30)):
    plot_case(st, i, save=os.path.join(d, f"row{i:04d}.png"))
print("saved plots to", d)
